In [1]:
import sys
sys.path.append("../src")

import os
from typing import Dict, Any

import torch
import numpy as np
import pandas as pd
import torch.nn as nn

from utils import *

In [2]:
data_path = os.path.join("..", "data", "raw", "gene-expression-normalized.csv")

df = pd.read_csv(data_path)

df.head()

,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.122203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.152391,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.160657,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.161598,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df.shape

(561, 19264)

In [4]:
df.iloc[0, :]

A1BG       0.0
A1CF       0.0
A2M        0.0
A2ML1      0.0
A3GALT2    0.0
          ... 
ZYG11A     0.0
ZYG11B     0.0
ZYX        0.0
ZZEF1      0.0
ZZZ3       0.0
Name: 0, Length: 19264, dtype: float64

In [5]:
pretrain_gene_x = torch.tensor(df.iloc[0, :].values).unsqueeze(0)

data_gene_ids = torch.arange(19264).repeat(pretrain_gene_x.shape[0], 1)

In [6]:
data_gene_ids

tensor([[    0,     1,     2,  ..., 19261, 19262, 19263]])

In [7]:
value_labels = pretrain_gene_x > 0

value_num = value_labels.sum(1)

max(value_num)

tensor(637)

In [7]:
value_labels.shape

torch.Size([1, 19264])

In [9]:
fake_data = torch.full((pretrain_gene_x.shape[0], max(value_num)), 103)

fake_data.shape

torch.Size([1, 637])

In [10]:
data = torch.hstack([pretrain_gene_x, fake_data])

data.shape

torch.Size([1, 19901])

In [11]:
fake_label = torch.full((value_labels.shape[0], max(value_num)), 1)

fake_label.shape

torch.Size([1, 637])

In [12]:
tmp_data = torch.tensor([(i + 1) * 20000 for i in range(value_labels.shape[1], 0, -1)])

tmp_data.shape

torch.Size([19264])

In [13]:
tmp_data

tensor([385300000, 385280000, 385260000,  ...,     80000,     60000,
            40000])

In [14]:
labels = value_labels + tmp_data

labels

tensor([[385300000, 385280000, 385260000,  ...,     80000,     60000,
             40000]])

In [15]:
labels.shape

torch.Size([1, 19264])

In [16]:
labels = torch.hstack([labels, fake_label])

labels.shape

torch.Size([1, 19901])

In [17]:
value_labels.shape

torch.Size([1, 19264])

In [18]:
fake_label_gene_idx = labels.topk(max(value_num)).indices

fake_label_gene_idx.shape

torch.Size([1, 637])

In [19]:
torch.gather(data, 1, fake_label_gene_idx).shape

torch.Size([1, 637])

In [20]:
data

tensor([[  0.,   0.,   0.,  ..., 103., 103., 103.]], dtype=torch.float64)

In [21]:
model_path = os.path.join("..", "assets", "models", "models.ckpt")

os.path.isfile(model_path)

True

In [22]:
model, config = load_model_frommmf(model_path)

{'mask_gene_name': False, 'gene_num': 19266, 'seq_len': 19266, 'encoder': {'hidden_dim': 768, 'depth': 12, 'heads': 12, 'dim_head': 64, 'seq_len': 19266, 'module_type': 'transformer', 'norm_first': False}, 'decoder': {'hidden_dim': 512, 'depth': 6, 'heads': 8, 'dim_head': 64, 'module_type': 'performer', 'seq_len': 19266, 'norm_first': False}, 'n_class': 104, 'pad_token_id': 103, 'mask_token_id': 102, 'bin_num': 100, 'bin_alpha': 1.0, 'rawcount': True, 'model': 'mae_autobin', 'test_valid_train_idx_dict': '/nfs_beijing/minsheng/data/os10000w-new/global_shuffle/meta.csv.train_set_idx_dict.pt', 'valid_data_path': '/nfs_beijing/minsheng/data/valid_count_10w.npz', 'num_tokens': 13, 'train_data_path': None, 'isPanA': False, 'isPlanA1': False, 'max_files_to_load': 5, 'bin_type': 'auto_bin', 'value_mask_prob': 0.3, 'zero_mask_prob': 0.03, 'replace_prob': 0.8, 'random_token_prob': 0.1, 'mask_ignore_token_ids': [0], 'decoder_add_zero': True, 'mae_encoder_max_seq_len': 15000, 'isPlanA': False, 'ma

In [23]:
config

{'mask_gene_name': False,
 'gene_num': 19266,
 'seq_len': 19266,
 'encoder': {'hidden_dim': 768,
  'depth': 12,
  'heads': 12,
  'dim_head': 64,
  'seq_len': 19266,
  'module_type': 'transformer',
  'norm_first': False},
 'decoder': {'hidden_dim': 512,
  'depth': 6,
  'heads': 8,
  'dim_head': 64,
  'module_type': 'performer',
  'seq_len': 19266,
  'norm_first': False},
 'n_class': 104,
 'pad_token_id': 103,
 'mask_token_id': 102,
 'bin_num': 100,
 'bin_alpha': 1.0,
 'rawcount': True,
 'model': 'mae_autobin',
 'test_valid_train_idx_dict': '/nfs_beijing/minsheng/data/os10000w-new/global_shuffle/meta.csv.train_set_idx_dict.pt',
 'valid_data_path': '/nfs_beijing/minsheng/data/valid_count_10w.npz',
 'num_tokens': 13,
 'train_data_path': None,
 'isPanA': False,
 'isPlanA1': False,
 'max_files_to_load': 5,
 'bin_type': 'auto_bin',
 'value_mask_prob': 0.3,
 'zero_mask_prob': 0.03,
 'replace_prob': 0.8,
 'random_token_prob': 0.1,
 'mask_ignore_token_ids': [0],
 'decoder_add_zero': True,
 'mae_

In [24]:
model

MaeAutobin(
  (token_emb): AutoDiscretizationEmbedding2(
    (mlp): Linear(in_features=1, out_features=100, bias=True)
    (mlp2): Linear(in_features=100, out_features=100, bias=True)
    (LeakyReLU): LeakyReLU(negative_slope=0.1)
    (Softmax): Softmax(dim=-1)
    (emb): Embedding(100, 768)
    (emb_mask): Embedding(1, 768)
    (emb_pad): Embedding(1, 768)
  )
  (pos_emb): Embedding(19267, 768)
  (decoder_embed): Linear(in_features=768, out_features=512, bias=True)
  (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (to_final): Linear(in_features=512, out_features=1, bias=True)
  (encoder): pytorchTransformerModule(
    (transformer_encoder): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=3072, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
     

In [6]:
class scFoundation(nn.Module):
    def __init__(
        self, 
        backbone: torch.nn,
        config: Dict[str, Any]
        ):

        self.backbone = backbone
        self.config = config
    
    def forward(self, x):
        if self.config["rawcount"] == False:
            

